# **Fake News Detection using NLP (Bag of N-grams)**

In [1]:
import pandas as pd
import spacy
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

In [2]:
df=pd.read_csv('Fake_Real_Data.csv')
print(df.head())
print(df.label.value_counts())

                                                Text label
0   Top Trump Surrogate BRUTALLY Stabs Him In The...  Fake
1  U.S. conservative leader optimistic of common ...  Real
2  Trump proposes U.S. tax overhaul, stirs concer...  Real
3   Court Forces Ohio To Allow Millions Of Illega...  Fake
4  Democrats say Trump agrees to work on immigrat...  Real
label
Fake    5000
Real    4900
Name: count, dtype: int64


In [3]:
nlp=spacy.load("en_core_web_sm")

def preprocess(text):
  doc=nlp(text)
  filtered_tokens=[]

  for token in doc:
    if token.is_stop or token.is_punct:
      continue
    filtered_tokens.append(token.lemma_)
  return ' '.join(filtered_tokens)

In [4]:
df['new_Text']=df.Text.apply(preprocess)
df.head(5)


,Text,label,new_Text
0,Top Trump Surrogate BRUTALLY Stabs Him In The...,Fake,Trump surrogate BRUTALLY Stabs Pathetic VIDE...
1,U.S. conservative leader optimistic of common ...,Real,U.S. conservative leader optimistic common gro...
2,"Trump proposes U.S. tax overhaul, stirs concer...",Real,trump propose U.S. tax overhaul stir concern d...
3,Court Forces Ohio To Allow Millions Of Illega...,Fake,Court Forces Ohio allow Millions illegally p...
4,Democrats say Trump agrees to work on immigrat...,Real,Democrats Trump agree work immigration bill wa...


In [11]:
print(len(df.loc[1,'Text']))
print(len(df.loc[1,'new_Text']))

758
535


In [18]:
X_train,X_test,y_train,y_test=train_test_split(df.new_Text,df.label,test_size=0.2, random_state=42,stratify=df.label)

print(y_train.value_counts())
print(y_test.value_counts())

label
Fake    4000
Real    3920
Name: count, dtype: int64
label
Fake    1000
Real     980
Name: count, dtype: int64


## **Naive Bayes**

In [19]:
model1=Pipeline([
    ('cv',CountVectorizer(ngram_range=(1,2))),
    ('nb',MultinomialNB())
])

model1.fit(X_train,y_train)
y_pred=model1.predict(X_test)
report=classification_report(y_test,y_pred)
print(report)

              precision    recall  f1-score   support

        Fake       0.98      0.97      0.98      1000
        Real       0.97      0.98      0.98       980

    accuracy                           0.98      1980
   macro avg       0.98      0.98      0.98      1980
weighted avg       0.98      0.98      0.98      1980



## **Random Forest**

In [20]:
model2=Pipeline([
    ('cv',CountVectorizer(ngram_range=(1,2))),
    ('nb',RandomForestClassifier())
])

model2.fit(X_train,y_train)
y_pred=model2.predict(X_test)
report=classification_report(y_test,y_pred)
print(report)

              precision    recall  f1-score   support

        Fake       0.99      1.00      0.99      1000
        Real       1.00      0.99      0.99       980

    accuracy                           0.99      1980
   macro avg       0.99      0.99      0.99      1980
weighted avg       0.99      0.99      0.99      1980



## **Conclusion:**
### In this project, Multinomial Naive Bayes and Random Forest were trained on preprocessed text using Bag of n-grams (unigram and bigram). Both models performed well, with Random Forest achieving slightly higher accuracy and F1-score.